In [ ]:
# ===============================
# IMPORT LIBRARIES
# ===============================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd


# ===============================
# LOAD DATASETS
# ===============================

casualties = pd.read_csv("dft-road-casualty-statistics-casualty-1979-latest-published-year.csv")
collisions = pd.read_csv("dft-road-casualty-statistics-collision-1979-latest-published-year.csv")
vehicles = pd.read_csv("dft-road-casualty-statistics-vehicle-1979-latest-published-year.csv")
la_ons_district_names = pd.read_csv("local-authority-ons-district-names.csv")


# ===============================
# SELECT VARIABLES + FILTER 2015+
# ===============================

casualties_1 = casualties[
    ["collision_ref_no","collision_year","vehicle_reference","casualty_reference",
     "sex_of_casualty","age_of_casualty","casualty_severity","casualty_type"]
]

casualties_1 = casualties_1[casualties_1["collision_year"] >= 2015]


collisions_1 = collisions[
    ["collision_ref_no","collision_year","longitude","latitude","date","day_of_week",
     "time","local_authority_district","local_authority_ons_district",
     "first_road_class","road_type","speed_limit","junction_detail",
     "junction_control","pedestrian_crossing","light_conditions",
     "urban_or_rural_area"]
]

collisions_1 = collisions_1[collisions_1["collision_year"] >= 2015]


vehicles_1 = vehicles[
    ["collision_ref_no","collision_year","vehicle_reference","vehicle_type"]
]

vehicles_1 = vehicles_1[vehicles_1["collision_year"] >= 2015]


# ===============================
# ADD LOCAL AUTHORITY NAMES
# ===============================

collisions_1 = collisions_1.merge(
    la_ons_district_names[["code","la_name"]],
    how="left",
    left_on="local_authority_ons_district",
    right_on="code"
)


# ===============================
# MERGE DATASETS
# ===============================

cas_coll = casualties_1.merge(
    collisions_1,
    how="left",
    on=["collision_ref_no","collision_year"]
)

full_data = cas_coll.merge(
    vehicles_1,
    how="left",
    on=["collision_ref_no","collision_year","vehicle_reference"]
)


# ===============================
# CREATE KSI DATASET
# ===============================

ksi_data = full_data[full_data["casualty_severity"].isin([1,2])]


# ===============================
# KEY VARIABLES
# ===============================

key_cols = [
"collision_year","longitude","latitude","date","day_of_week","time",
"local_authority_district","first_road_class","road_type","speed_limit",
"junction_detail","junction_control","pedestrian_crossing",
"light_conditions","urban_or_rural_area","vehicle_type"
]

ksi_data_code_match = ksi_data.dropna(subset=key_cols)


# ===============================
# REPLACE -1 WITH NaN
# ===============================

ksi_data = ksi_data.replace(-1, np.nan)
ksi_data_code_match = ksi_data_code_match.replace(-1, np.nan)


# ===============================
# MISSING VALUE SUMMARY
# ===============================

na_summary = ksi_data.isna().sum().reset_index()
na_summary.columns = ["column","na_count"]

na_summary["total_rows"] = len(ksi_data)
na_summary["na_percent"] = (na_summary["na_count"]/len(ksi_data))*100

print(na_summary)


na_summary_code_match = ksi_data_code_match.isna().sum().reset_index()
na_summary_code_match.columns = ["column","na_count"]

na_summary_code_match["total_rows"] = len(ksi_data_code_match)
na_summary_code_match["na_percent"] = (na_summary_code_match["na_count"]/len(ksi_data_code_match))*100

print(na_summary_code_match)


# ===============================
# KSI TRENDS
# ===============================

ksi_by_year = ksi_data.groupby("collision_year").size().reset_index(name="KSI_count")

fatalities = ksi_data[ksi_data["casualty_severity"]==1]

fatalities_by_year = fatalities.groupby("collision_year").size().reset_index(name="fatalities_count")


# ===============================
# PLOT KSI TREND
# ===============================

plt.figure()

plt.plot(
    ksi_by_year["collision_year"],
    ksi_by_year["KSI_count"],
    marker="o"
)

plt.title("Trend of KSI (Killed or Seriously Injured) Over Years")
plt.xlabel("Year")
plt.ylabel("Number of KSI")

plt.show()


# ===============================
# PLOT FATALITY TREND
# ===============================

plt.figure()

plt.plot(
    fatalities_by_year["collision_year"],
    fatalities_by_year["fatalities_count"],
    marker="o"
)

plt.title("Trend of Fatalities Over Years")
plt.xlabel("Year")
plt.ylabel("Number of Fatalities")

plt.show()


# ===============================
# LOAD MAP DATA
# ===============================

uk = gpd.read_file("UK_Boundaries.shp")
ireland = gpd.read_file("Ireland_Boundaries.shp")

uk_ir_shp_df = pd.concat([uk, ireland])


# ===============================
# MAP PLOT
# ===============================

fig, ax = plt.subplots()

uk_ir_shp_df.plot(ax=ax, color="lightblue")

plt.title("UK and Ireland Map")

plt.show()


# ===============================
# MAP WITH COLLISION POINTS
# ===============================

fig, ax = plt.subplots()

uk_ir_shp_df.plot(ax=ax, color="snow")

plt.scatter(
    ksi_data["longitude"],
    ksi_data["latitude"],
    s=1,
    alpha=0.1
)

plt.show()


# ===============================
# COLLISIONS BY ROAD TYPE
# ===============================

coll_by_road_type = collisions_1.groupby("road_type").size().reset_index(name="num_collisions")

sns.barplot(data=coll_by_road_type, x="road_type", y="num_collisions")

plt.title("Collisions by Road Type")

plt.xticks(rotation=45)

plt.show()


# ===============================
# COLLISIONS BY SPEED LIMIT
# ===============================

coll_by_speed = collisions_1.groupby("speed_limit").size().reset_index(name="num_collisions")

sns.barplot(data=coll_by_speed, x="speed_limit", y="num_collisions")

plt.title("Collisions by Speed Limit")

plt.show()


# ===============================
# COLLISIONS BY DAY OF WEEK
# ===============================

coll_by_day = collisions_1.groupby("day_of_week").size().reset_index(name="num_collisions")

sns.barplot(data=coll_by_day, x="day_of_week", y="num_collisions")

plt.title("Collisions by Day of Week")

plt.show()


# ===============================
# COLLISIONS BY JUNCTION TYPE
# ===============================

coll_by_junction = collisions_1.groupby("junction_detail").size().reset_index(name="num_collisions")

sns.barplot(data=coll_by_junction, x="junction_detail", y="num_collisions")

plt.title("Collisions by Junction Type")

plt.xticks(rotation=45)

plt.show()


# ===============================
# LOCAL AUTHORITY HOTSPOTS
# ===============================

collisions_clean_LA = collisions_1[
    (collisions_1["local_authority_ons_district"].notna()) &
    (collisions_1["local_authority_ons_district"] != -1)
]


hotspot_districts = (
    collisions_clean_LA
    .groupby("la_name")
    .size()
    .reset_index(name="num_collisions")
    .sort_values("num_collisions", ascending=False)
)


top10_districts = hotspot_districts.head(10)


sns.barplot(
    data=top10_districts,
    x="num_collisions",
    y="la_name"
)

plt.title("Top 10 Local Authority Districts by Number of Collisions (2015–2024)")
plt.xlabel("Number of Collisions")
plt.ylabel("Local Authority")

plt.show()